# Second Dataset Experiment
- **Group Name**: MVPs
- **Course:** Advanced Deep Learning - AIGC-5500-0NA
- **Professor**: Hossein Pourmodheji
- **Owner:** Abhi Patel
---


Starter notebook: shared data loading and batch verification are provided; model implementation, training, and results remain to be completed. No results are fabricated.

Retrain the baseline, CNN, LSTM, and Transformer from scratch on a distinct dataset with three seeds each. Fit a separate vocabulary on that dataset's training partition.


## 1. Initial Configuration


### 1.1. Imports, Paths, and Device


In [ ]:
# Import required libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

# Locate the repository from either its root or the notebooks folder
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'src').is_dir() else current_dir.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Place this notebook in the repository notebooks folder.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project modules
from src.config import BATCH_SIZE, TRAINING_SEEDS
from src.data import load_prepared_data
from src.reproducibility import set_seed
from src.text import EncodedTextDataset, make_data_loader

# Use CUDA when it is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the runtime and selected device
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

### 1.2. Experiment Configuration


In [ ]:
# Name this experiment and its owner
MODEL_NAME = 'second_dataset'
OWNER = 'Abhi Patel'

# Reset randomness before the first run
set_seed(TRAINING_SEEDS[0])

# Set these only after selecting and preparing a different dataset
SECOND_DATASET_ID = None
PREPARED_DATA_DIR = None
RESULTS_DIR = None
CHECKPOINT_DIR = None

# Stop here until the second dataset is configured
print('Second dataset must be selected and prepared before section 2.')

## 2. Load Shared Prepared Data


Prepare this dataset separately using the shared cleaning/tokenization policy and save the same artifact schema. Do not reuse the Enron vocabulary. The guard below intentionally stops execution until a distinct dataset is configured.


In [ ]:
# Require a prepared directory for a dataset other than Enron
if SECOND_DATASET_ID is None or PREPARED_DATA_DIR is None:
    raise ValueError('Configure a distinct second dataset and its prepared-data directory first.')
if SECOND_DATASET_ID == 'SetFit/enron_spam':
    raise ValueError('The second dataset must differ from Enron.')

# Point results and checkpoints at the second-dataset folders
PREPARED_DATA_DIR = Path(PREPARED_DATA_DIR)
RESULTS_DIR = PROJECT_ROOT / 'results' / 'second_dataset'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'second_dataset'

In [ ]:
# Load the saved splits, vocabulary, and metadata
prepared = load_prepared_data(PREPARED_DATA_DIR)

# Keep the cleaned partitions
splits = prepared['splits']

# Keep the training vocabulary
vocabulary = prepared['vocabulary']

# Keep the preprocessing configuration
metadata = prepared['metadata']

# Use the saved sequence length instead of choosing a new one
MAX_LENGTH = metadata['max_length']

# Confirm the vocabulary size and reserved token IDs
assert len(vocabulary) == metadata['vocabulary_size']
assert vocabulary['<pad>'] == 0 and vocabulary['<unk>'] == 1

# Confirm each split still has the saved number of rows
for name, expected_size in metadata['split_sizes'].items():
    assert len(splits[name]) == expected_size

# Show the loaded split sizes
pd.DataFrame([{'split': name, 'rows': len(part)} for name, part in splits.items()])

In [ ]:
# Confirm the saved metadata names the selected dataset
if metadata['dataset_id'] != SECOND_DATASET_ID:
    raise ValueError('Prepared data does not match the selected second dataset.')

In [ ]:
# Print dataset information
print('Vocabulary size:', len(vocabulary))
print('Maximum length:', MAX_LENGTH)
print('Dataset:', metadata['dataset_id'])
print('Dataset revision:', metadata['dataset_revision'])

### 2.1. Encode Development Partitions


Only training and validation are encoded. Recreate the training loader for each seed; keep validation order fixed.


In [ ]:
# Encode train and validation only
encoded_datasets = {
    name: EncodedTextDataset(
        texts = splits[name]['text'],
        labels = splits[name]['label'],
        vocabulary = vocabulary,
        max_length = MAX_LENGTH,
        ) for name in ('train', 'validation')
    }

# Keep validation order fixed
validation_loader = make_data_loader(encoded_datasets['validation'], batch_size = BATCH_SIZE, shuffle = False, seed = TRAINING_SEEDS[0])

# Inspect one validation batch
example_batch = next(iter(validation_loader))
for name, tensor in example_batch.items():
    print(name, tuple(tensor.shape))

# Recreate the training loader inside every seeded run, before training.
# Do not encode or evaluate the test partition during development.